# Lab 1.1 - Train and Evaluate a Classifier (Solution)
**Day 1 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day1/lab1_1_classifier/solution/lab1_1_classifier.ipynb)

**Goal:** Train a classifier on a small tabular dataset, evaluate it with the
right metrics, and see overfitting with your own eyes.

This lab runs on a Colab CPU runtime in under a minute. No GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; most are preinstalled on Colab).
%pip install -q scikit-learn matplotlib pandas numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 1. Load the data
The breast cancer dataset is a small, labeled tabular dataset: 30 numeric
features per sample and a binary label (malignant or benign).

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Samples:", X.shape[0], "| Features:", X.shape[1])
print("Classes:", dict(zip(data.target_names, np.bincount(y))))

## 2. Split into train and test sets
We hold out 25% of the data for testing. The model never sees it during
training, so the test score is an honest estimate of real-world performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

## 3. Train a classifier
We scale the features first (logistic regression works best on standardized
inputs), then fit the model on the training set only.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
clf.fit(X_train_scaled, y_train)
print("Training complete.")

## 4. Evaluate with the right metrics
Accuracy alone can mislead on imbalanced data, so we also report precision,
recall, and F1.

In [ ]:
preds = clf.predict(X_test_scaled)
print(f"Accuracy:  {accuracy_score(y_test, preds):.3f}")
print(f"Precision: {precision_score(y_test, preds):.3f}")
print(f"Recall:    {recall_score(y_test, preds):.3f}")
print(f"F1 score:  {f1_score(y_test, preds):.3f}")
print("Confusion matrix:\n", confusion_matrix(y_test, preds))

## 5. See overfitting in action
A decision tree grown with no depth limit can memorize the training data.
Watch the gap between its train and test accuracy.

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
deep_tree.fit(X_train, y_train)
train_acc = accuracy_score(y_train, deep_tree.predict(X_train))
test_acc = accuracy_score(y_test, deep_tree.predict(X_test))
print(f"Deep tree  -> train: {train_acc:.3f}  test: {test_acc:.3f}  (big gap = overfit)")

shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
shallow_tree.fit(X_train, y_train)
train_acc_s = accuracy_score(y_train, shallow_tree.predict(X_train))
test_acc_s = accuracy_score(y_test, shallow_tree.predict(X_test))
print(f"Shallow tree -> train: {train_acc_s:.3f}  test: {test_acc_s:.3f}  (smaller gap)")

## Stretch goals
1. Sweep tree depth from 1 to 15 and plot train vs test accuracy to find where
   overfitting begins.
2. Try a `RandomForestClassifier` and compare its F1 to logistic regression.

In [ ]:
depths = range(1, 16)
train_scores, test_scores = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test, tree.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(list(depths), train_scores, marker="o", label="train")
plt.plot(list(depths), test_scores, marker="o", label="test")
plt.xlabel("tree depth")
plt.ylabel("accuracy")
plt.title("Train vs test accuracy as the tree gets deeper")
plt.legend()
plt.show()

### Expected output
- Logistic regression reaches about 0.97-0.99 accuracy with matching
  precision, recall, and F1.
- The deep tree scores 1.00 on training but noticeably lower on test data,
  the classic overfitting gap.
- In the stretch plot, train accuracy climbs to 1.0 while test accuracy
  plateaus, showing where extra depth stops helping.